# Глава 5. Предварительное обучение на неразмеченных данных

In [1]:
pip install matplotlib numpy tiktoken torch tensorflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from importlib.metadata import version

pkgs = ["matplotlib", 
        "numpy", 
        "tiktoken", 
        "torch",
        "tensorflow" # Для предварительно обученных моделей OpenAI
       ]
for p in pkgs:
    print(f"{p} Версия: {version(p)}")

matplotlib Версия: 3.10.9
numpy Версия: 2.4.4
tiktoken Версия: 0.12.0
torch Версия: 2.12.0
tensorflow Версия: 2.21.0


- В этой главе мы реализуем цикл обучения и код для базовой оценки модели, чтобы провести предварительное обучение большой языковой модели
- В конце мы также загружаем в нашу модель общедоступные предварительно обученные веса от OpenAI

<img src="https://camo.githubusercontent.com/137f57f6192fbcb6627e6ced1b5274c71924774dec18a4dea29b1c156619ef24/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f30312e77656270" width=800px>

- Ниже перечислены темы, затронутые в этой главе

<img src="https://camo.githubusercontent.com/01ebc99e37dddc617ba6dd20799f945fd6a562bac8b8abe5a4b82eb491ebbfca/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f30322e77656270" width=800px>

&nbsp;
## 5.1 Оценка генеративных текстовых моделей

- В начале этого раздела мы кратко расскажем о том, как инициализировать модель GPT с помощью кода из предыдущей главы
- Затем мы обсудим основные метрики оценки больших языковых моделей
- Наконец, в этом разделе мы применим эти метрики оценки к обучающему и проверочному наборам данных

&nbsp;
### 5.1.1 Использование GPT для генерации текста

- Мы инициализируем модель GPT с помощью кода из предыдущей главы

In [3]:
import torch
from previous_chapters import GPTModel
# Если файл `previous_chapters.py` недоступен локально,
# вы можете импортировать его из пакета PyPI `llms-from-scratch`. 
# Подробнее см.: https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# Например,
# from llms_from_scratch.ch04 import GPTModel

GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Размер словаря
    "context_length": 256, # Сокращенная длина контекста (исходное значение: 1024)
    "emb_dim": 768,        # Размерность эмбеддинга
    "n_heads": 12,         # Количество ядер внимания
    "n_layers": 12,        # Количество слоев
    "drop_rate": 0.1,      # Коэффициент дропаута
    "qkv_bias": False      # Смещение в сторону значений ключей запроса
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();  # Отключите дропаут во время логического вывода

- Мы используем дропаут 0,1, но в настоящее время довольно часто обучают большие языковые модели без дропаута
- В современных больших языковых моделях также не используются векторы смещения в слоях `nn.Linear` для матриц запросов, ключей и значений (в отличие от более ранних моделей GPT). Это достигается за счет установки параметра `"qkv_bias": False`
- Мы уменьшили длину контекста (`context_length`) всего на 256 токенов, чтобы снизить требования к вычислительным ресурсам для обучения модели, в то время как исходная модель GPT-2 со 124 миллионами параметров использовала 1024 токена
    - Это сделано для того, чтобы мы могли следить за примерами кода и выполнять их на своих портативных компьютерах
    - Позже мы также загрузим модель с `context_length` 1024 из предварительно обученных весов.

- Далее мы используем функцию `generate_text_simple` из предыдущей главы для генерации текста
- Кроме того, мы определяем две вспомогательные функции: `text_to_token_ids` и `token_ids_to_text` — для преобразования токенов в текстовое представление и обратно, которые мы будем использовать на протяжении всей главы

<img src="https://camo.githubusercontent.com/0756c65e7e7878cab7b7673bedd3f441cf42f1f67e89b46d9e7ec931e0fffbc3/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f30332e77656270" width=800px>

In [4]:
import tiktoken
from previous_chapters import generate_text_simple

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # добавить размер пакета
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # удалить размер пакета
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Выводимый текст:\n", token_ids_to_text(token_ids, tokenizer))

Выводимый текст:
 Every effort moves you rentingetic wasnم refres RexMeCHicular stren


- Как мы видим выше, модель не генерирует качественный текст, потому что она еще не обучена
- Как измерить или зафиксировать в числовом выражении, что такое «качественный текст», чтобы отслеживать этот показатель во время обучения?
- В следующем подразделе мы рассмотрим метрики для расчета показателя потерь для сгенерированных результатов, которые можно использовать для оценки прогресса обучения
- В следующих главах, посвященных тонкой настройке больших языковых моделей, мы также рассмотрим дополнительные способы оценки качества модели

&nbsp;
### 5.1.2 Расчет потерь при генерации текста: кросс-энтропия и перплексия

- Предположим, у нас есть тензор `inputs`, содержащий идентификаторы токенов для двух обучающих примеров (строк)
- Соответствующие `inputs`, `targets` содержат желаемые идентификаторы токенов, которые мы хотим, чтобы модель сгенерировала
- Обратите внимание, что `targets` — это `inputs`, сдвинутые на одну позицию

In [5]:
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",Ф
                        [1107,  588, 11311]]) #  " really like chocolate"]

- Подавая на вход модели данные, мы получаем вектор логитов для двух входных примеров, каждый из которых состоит из 3 токенов
- Каждый токен представляет собой вектор из 50 257 элементов, соответствующий размеру словаря
- Применяя функцию softmax, мы можем преобразовать тензор логитов в тензор той же размерности, содержащий оценки вероятности

In [6]:
with torch.no_grad():
    logits = model(inputs)

probas = torch.softmax(logits, dim=-1) # Вероятность появления каждого токена в словаре
print(probas.shape) # Форма: (размер пакета, количество токенов, размер словаря)

torch.Size([2, 3, 50257])


- На рисунке ниже с использованием очень небольшого набора слов для наглядности показано, как мы преобразуем оценки вероятности обратно в текст

<img src="https://camo.githubusercontent.com/f98790fc96dfefdd3e61533ca406f241a893976c8cb7668afbcec178763c45a0/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f30342e77656270" width=800px>

- Ммы можем использовать функцию `argmax`, чтобы преобразовать значения вероятности в предсказанные идентификаторы токенов
- Функция `softmax`, описанная выше, сгенерировала 50 257-мерный вектор для каждого токена. Функция `argmax` возвращает позицию с наибольшим значением вероятности в этом векторе, которая и является предсказанным идентификатором токена

- Поскольку у нас есть 2 входных пакета по 3 токена в каждом, мы получаем 2 на 3 предсказанных идентификатора токенов:

In [7]:
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print("Идентификаторы токенов:\n", token_ids)

Идентификаторы токенов:
 tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [41751]]])


- Если мы расшифруем эти токены, то увидим, что они сильно отличаются от тех, которые мы хотим, чтобы модель предсказывала

In [8]:
print(f"Целевой batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Фактический batch 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

Целевой batch 1:  effort moves you
Фактический batch 1:  Armed heNetflix


- Это потому, что модель еще не обучена
- Чтобы обучить модель, нам нужно знать, насколько она далека от правильных прогнозов (целевых значений)

<img src="https://camo.githubusercontent.com/3dee5bf33ad015c683fa2a9a91ae611b263101027fa2f9415934ae1a1c38778e/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f30362e77656270" width=800px>

- Вероятности появления токенов, соответствующие целевым индексам, следующие:

In [9]:
text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Текст 1:", target_probas_1)

text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Текст 2:", target_probas_2)

Текст 1: tensor([7.4541e-05, 3.1061e-05, 1.1563e-05])
Текст 2: tensor([1.0337e-05, 5.6776e-05, 4.7559e-06])


 ---

### Код выше

Эта строка выполняет **индексирование многомерного массива (тензора) `probas`** для извлечения вероятностей, соответствующих правильным целевым классам.

### Пошаговый разбор

### 1. `text_idx`
Скалярный индекс, указывающий на конкретный текст в батче.
- `probas[text_idx, ...]` → выбирает двумерный срез формы `(количество_токенов, количество_классов)` для одного текста.

### 2. `[0, 1, 2]`
Список индексов токенов.
- `probas[text_idx, [0, 1, 2], ...]` → выбирает строки только для токенов с индексами 0, 1, 2 (первые три токена).
- После этого измерения получается форма `(3, количество_классов)`.

### 3. `targets[text_idx]`
Одномерный массив формы `(количество_токенов,)`, содержащий **истинные метки классов** для каждого токена в тексте `text_idx`.
- `targets[text_idx]` → вектор правильных классов для всего текста.
- Но поскольку на предыдущем шаге мы выбрали только токены `[0, 1, 2]`, здесь **неявно тоже берутся первые три элемента** этого вектора (благодаря broadcasting/advanced indexing).

### 4. Полное индексирование
```python
probas[text_idx, [0, 1, 2], targets[text_idx]]
```
NumPy/PyTorch выполняет **advanced indexing**:
- Для каждого из выбранных токенов (0, 1, 2) извлекается вероятность того класса, который указан в `targets` для этого же токена.
- Фактически это эквивалентно:
```python
[
    probas[text_idx, 0, targets[text_idx][0]],  # вер-ть правильного класса для токена 0
    probas[text_idx, 1, targets[text_idx][1]],  # вер-ть правильного класса для токена 1
    probas[text_idx, 2, targets[text_idx][2]]   # вер-ть правильного класса для токена 2
]
```

### Результат

**`target_probas_2`** — одномерный массив из трёх чисел (вероятностей), показывающих, насколько модель была уверена в **правильных** классах для первых трёх токенов конкретного текста.

Это часто используется для:
- анализа уверенности модели в правильных ответах,
- вычисления **confidence** срезов,
- поиска сложных примеров (где верная вероятность мала),
- отладки качества предсказаний на уровне отдельных токенов.

---

- Мы хотим максимизировать все эти значения, приблизив их к вероятности 1.
- В математической оптимизации проще максимизировать логарифм показателя вероятности, чем сам показатель вероятности. Лекция с более подробным описанием: [L8.2 Функция потерь логистической регрессии](https://www.youtube.com/watch?v=GxJe0DZvydM)

In [12]:
# Вычислить логарифм всех вероятностей токенов
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
print(log_probas)

tensor([ -9.5042, -10.3796, -11.3677, -11.4798,  -9.7764, -12.2561])


- Далее мы вычисляем среднюю логарифмическую вероятность:

In [13]:
# Рассчитайте среднюю вероятность для каждого токена
avg_log_probas = torch.mean(log_probas)
print(avg_log_probas)

tensor(-10.7940)


- Цель состоит в том, чтобы сделать эту среднюю логарифмическую вероятность как можно более высокой за счет оптимизации весовых коэффициентов модели
- Из-за логарифмической функции максимально возможное значение равно 0, а мы пока далеки от этого значения

- В глубоком обучении вместо максимизации средней логарифмической вероятности принято минимизировать *отрицательное* значение средней логарифмической вероятности. В нашем случае вместо того, чтобы максимизировать -10.7940, чтобы оно приблизилось к 0, в глубоком обучении мы минимизируем -10.7940, чтобы оно приблизилось к 0
- Отрицательное значение -10.7940, то есть -10.7940, в глубоком обучении также называют кросс-энтропийной потерей

In [14]:
neg_avg_log_probas = avg_log_probas * -1
print(neg_avg_log_probas)

tensor(10.7940)


В PyTorch уже реализована функция `cross_entropy`, которая выполняет описанные выше действия

<img src="https://camo.githubusercontent.com/3d6fb2ee91bebb246351570506a344d7d579fc161084faec6856c0659c815452/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f30372e77656270" width=800px>

- Прежде чем применить функцию `cross_entropy`, давайте проверим форму логитов и целевых значений

In [15]:
# Логиты имеют форму (batch_size, num_tokens, vocab_size)
print("Форма логитов:", logits.shape)

# Цель имеет форму (batch_size, num_tokens)
print("Целевая форма:", targets.shape)

Форма логитов: torch.Size([2, 3, 50257])
Целевая форма: torch.Size([2, 3])


- Для функции `cross_entropy` в PyTorch мы хотим сгладить эти тензоры, объединив их по размерности пакета:

In [16]:
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()

print("Сглаженные логиты:", logits_flat.shape)
print("Сглаженные цели:", targets_flat.shape)

Сглаженные логиты: torch.Size([6, 50257])
Сглаженные цели: torch.Size([6])


- Обратите внимание, что целевыми значениями являются идентификаторы токенов, которые также представляют собой позиции индексов в тензорах логитов, которые мы хотим максимизировать
- Функция `cross_entropy` в PyTorch автоматически применяет функцию `softmax` и вычисляет логарифмическую вероятность для тех индексов токенов в логитах, которые нужно максимизировать

In [17]:
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(loss)

tensor(10.7940)


- Понятие, связанное с кросс-энтропийной потерей, — это перплексия большой языковой модели
- Перплексия — это просто экспоненциальная функция от кросс-энтропийной потери

In [18]:
perplexity = torch.exp(loss)
print(perplexity)

tensor(48725.8203)


- Показатель перплексии часто считается более интерпретируемым, поскольку его можно рассматривать как эффективный размер словаря, в котором модель не уверена на каждом этапе (в приведенном выше примере это 48 725 слов или токенов)
- Другими словами, перплексия показывает, насколько хорошо распределение вероятностей, предсказанное моделью, соответствует реальному распределению слов в наборе данных
- Как и в случае с логарифмической функцией потерь, чем ниже перплексия, тем ближе предсказания модели к реальному распределению

 ---

### Представим, что большая языковая модель — это **попугай**, который учится говорить.

### 🦜 1. Зачем нужна модель (попугай)
Представь, что у нас есть попугай, который пока не умеет говорить. Мы хотим научить его заканчивать фразы. Мы говорим: **"Каждое усилие двигает..."**, а попугай должен договорить: **"...тебя"**.

Но сначала попугай даже не знает слов и говорит полную чушь: **"Каждое усилие двигает крокодил летать фиолетовый"**.

### 📏 2. Как измерить, насколько плох попугай? (кросс-энтропия)
Нам нужна линейка, чтобы **измерить ошибки** попугая.

### Шаг 1: Попугай выдаёт вероятности
Когда мы говорим слово, попугай-модель не выдаёт сразу одно слово — он выдаёт **список всех слов** с их вероятностями. Например:
- "тебя" — вероятность 0.0001 (очень маленькая, он почти не верит в это слово)
- "крокодил" — вероятность 0.8 (он очень верит, что дальше будет "крокодил")

### Шаг 2: Смотрим, какую вероятность попугай дал ПРАВИЛЬНОМУ слову
Мы знаем правильное слово — "тебя". Попугай дал ему вероятность 0.0001. Это **очень плохо**. Если бы он был умным, он дал бы вероятность 0.9999 (почти 1).

Код:
```python
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
```
**Перевод:** "Попугай, покажи, насколько ты веришь в правильные слова 'effort', 'moves', 'you'. Ага, веришь на 0.0001, 0.0002, 0.00005 — ужасно!"

### 🔢 3. Почему берут логарифм?
Числа вроде 0.0000001 неудобно складывать и сравнивать. **Логарифм** — это как специальное увеличительное стекло, которое превращает очень маленькие вероятности в "нормальные" отрицательные числа:
- 0.0001 → логарифм = -9.21
- 0.5 → логарифм = -0.69
- 0.999 → логарифм = -0.001

Чем ближе к 0, тем лучше. У идеального попугая логарифмы были бы 0.

### ✖️ 4. Зачем переворачивать знак? (кросс-энтропия)
В школе нас учат **минимизировать ошибки**, а не максимизировать успех. Поэтому мы берём логарифмы и **умножаем на -1**, чтобы перевернуть:
- Было: цель — чтобы логарифм стал 0 (максимум).
- Стало: цель — чтобы **отрицательный логарифм** стал 0 (минимум).

`neg_avg_log_probas = avg_log_probas * -1` — это как сказать: "Твоя ошибка сейчас 10.794. Уменьшай её до 0!"

Это число и есть **кросс-энтропийная потеря** — главная оценка, насколько сильно ошибается попугай.

### 🤯 5. Что такое перплексия? (простыми словами)
**Перплексия = exp(потеря)**.

Это число говорит: **"Попугай сейчас как будто выбирает из скольких слов?"**

- Если перплексия = **48,725** (как в примере) → попугай в панике, как будто перед ним словарь из 48 тысяч слов, и он гадает наугад.
- Если перплексия = **5** → попугай уже почти выучился, колеблется только между 5 похожими словами.
- Если перплексия = **1** → попугай идеально знает каждое слово.

### 🎯 Итог: зачем ВСЁ ЭТО?
Весь этот сложный код с вероятностями, логарифмами и перплексией нужен для одного:

**Чтобы компьютер мог САМ, без человека, понять, хорошо ли он учится говорить.**
- Смотрит на правильные слова
- Смотрит, какие вероятности он им дал
- Считает "ошибку" (кросс-энтропию)
- Старается уменьшить ошибку → учится говорить лучше

Это как если бы попугай сам себя проверял по учебнику и исправлял ошибки, пока не заговорит, как человек.

---

&nbsp;
### 5.1.3 Расчет потерь для обучающей и проверочной выборок

- Для обучения большой языковой модели мы используем относительно небольшой набор данных (по сути, всего одну короткую историю)
- Причины в следующем:
  - Вы можете запустить примеры кода за несколько минут на ноутбуке без подходящего графического процессора
  - Обучение завершается относительно быстро (за несколько минут, а не недель), что удобно для образовательных целей
  - Мы используем текст из общественного достояния, который можно включить в этот репозиторий на GitHub без нарушения авторских прав и без увеличения размера репозитория


- Например, для обучения Llama 2 7B на 2 триллионах токенов потребовалось 184 320 часов работы на графических процессорах A100
  - На момент написания этой статьи почасовая стоимость облачного сервера 8xA100 на AWS составляла примерно 30 долларов США
  - Таким образом, по приблизительным подсчетам, обучение этой большой языковой модели обойдется в 184 320 / 8 * 30 долларов США = 690 000 долларов США

In [19]:
import os
import requests

file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text_data = response.text
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()


# Изначально в книге использовался следующий код:
# Однако urllib использует более старые настройки протокола, которые
# могут вызвать проблемы у некоторых пользователей VPN. 
# Приведенная выше версия с использованием requests более надежна
# в этом отношении.

        
# import os
# import urllib.request

# file_path = "the-verdict.txt"
# url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

# if not os.path.exists(file_path):
#     with urllib.request.urlopen(url) as response:
#         text_data = response.read().decode('utf-8')
#     with open(file_path, "w", encoding="utf-8") as file:
#         file.write(text_data)
# else:
#     with open(file_path, "r", encoding="utf-8") as file:
#         text_data = file.read()

- Быстрая проверка корректности загрузки текста путем вывода на экран первых и последних 99 символов

In [20]:
# Первые 99 символов
print(text_data[:99])

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [21]:
# Последние 99 символов
print(text_data[-99:])

it for me! The Strouds stand alone, and happen once--but there's no exterminating our kind of art."


In [22]:
total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))

print("Количество символов:", total_characters)
print("Количество токенов:", total_tokens)

Количество символов: 20479
Количество токенов: 5145


- Текст состоит из 5145 токенов и слишком короток для обучения большой языковой модели, но, повторюсь, это делается в образовательных целях (позже мы также загрузим предварительно обученные модели)

- Далее мы делим набор данных на обучающую и проверочную выборки и с помощью загрузчиков данных из главы 2 подготавливаем пакеты для обучения большой языковой модели
- Для наглядности на рисунке ниже указано значение `max_length=6`, но для загрузчика обучающей выборки мы устанавливаем `max_length` равным длине контекста, поддерживаемой большой языковой моделью
- Для простоты на рисунке показаны только входные токены
    - Поскольку мы обучаем LLM предсказывать следующее слово в тексте, цели выглядят так же, как и эти входные данные, за исключением того, что цели сдвинуты на одну позицию

<img src="https://camo.githubusercontent.com/216592059168cdb0fba2d8b9843bfc411a1898c407163cabb1d98a59e7cc584a/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f30392e77656270" width=800px>

In [24]:
from previous_chapters import create_dataloader_v1

# Соотношение обучения и валидации
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]


torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [ ]:
# Проверка на вменяемость

if total_tokens * (train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Недостаточно токенов для загрузчика обучения. "
          "Попробуйте уменьшить `GPT_CONFIG_124M['context_length']` или "
          "увеличить `training_ratio`")

if total_tokens * (1-train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Недостаточно токенов для валидационного загрузчика. "
          "Попробуйте уменьшить `GPT_CONFIG_124M['context_length']` или "
          "снизить `training_ratio`")

- Мы используем относительно небольшой размер пакета, чтобы снизить нагрузку на вычислительные ресурсы, а также потому, что исходный набор данных очень мал
- Например, Llama 2 7B обучалась с размером пакета 1024

- Дополнительная проверка правильности загрузки данных:

In [27]:
print("Обучающий загрузчик:")
for x, y in train_loader:
    print(x.shape, y.shape)

print("\nПроверочный загрузчик:")
for x, y in val_loader:
    print(x.shape, y.shape)

Обучающий загрузчик:
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])

Проверочный загрузчик:
torch.Size([2, 256]) torch.Size([2, 256])


- Еще одна дополнительная проверка, подтверждающая, что размер токенов соответствует ожидаемому:

In [28]:
train_tokens = 0
for input_batch, target_batch in train_loader:
    train_tokens += input_batch.numel()

val_tokens = 0
for input_batch, target_batch in val_loader:
    val_tokens += input_batch.numel()

print("Обучающие токены:", train_tokens)
print("Проверочные токены:", val_tokens)
print("Все токены:", train_tokens + val_tokens)

Обучающие токены: 4608
Проверочные токены: 512
Все токены: 5120


- Далее мы реализуем вспомогательную функцию для расчета кросс-энтропийной потери для заданного пакета данных
- Кроме того, мы реализуем вторую вспомогательную функцию для расчета потерь для заданного пользователем количества пакетов в загрузчике данных

In [29]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        # Уменьшите количество пакетов до общего количества пакетов в загрузчике данных
        # если num_batches превышает количество пакетов в загрузчике данных
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

 ---

### Код выше

Представь, что ты учишь робота предсказывать следующее слово в тексте. Эти две функции — как учитель, который проверяет, насколько хорошо робот справляется с заданием.

### Первая функция: `calc_loss_batch` — "Проверка одной пачки заданий"

```python
def calc_loss_batch(input_batch, target_batch, model, device):
    # Перемещаем данные на "рабочий стол" (GPU или CPU)
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    
    # Даём роботу входные данные, он выдаёт свои догадки
    logits = model(input_batch)
    
    # Сравниваем догадки робота с правильными ответами
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    
    return loss  # Возвращаем "количество ошибок"
```

**Что происходит простыми словами:**
- Ты даёшь роботу **несколько примеров сразу** (пачку), чтобы он работал быстрее
- Робот смотрит на начало фразы и пытается угадать следующее слово
- Функция сравнивает догадки робота с правильными ответами
- Чем больше ошибок — тем больше число `loss` (потеря)
- Если робот угадал идеально — `loss` будет около 0

**Зачем нужно:**
- Чтобы понять, насколько хорошо робот учится
- Если `loss` уменьшается — робот становится умнее!

### Вторая функция: `calc_loss_loader` — "Проверка всей домашней работы"

```python
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.  # Корзинка для сбора всех ошибок
    
    # Если нет ни одного задания — говорим "не могу посчитать"
    if len(data_loader) == 0:
        return float("nan")
    
    # Сколько пачек заданий проверить?
    elif num_batches is None:
        num_batches = len(data_loader)  # Проверим все пачки
    else:
        num_batches = min(num_batches, len(data_loader))  # Проверим сколько просят
    
    # Проходим по пачкам заданий
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            # Проверяем очередную пачку и добавляем ошибки в корзинку
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()  # .item() превращает в обычное число
        else:
            break  # Проверили сколько нужно — останавливаемся
    
    # Возвращаем среднее количество ошибок на пачку
    return total_loss / num_batches
```

**Что происходит простыми словами:**
- Представь, что у тебя есть большая стопка тетрадей (это `data_loader`)
- Ты можешь проверить все тетради или только часть (это `num_batches`)
- Ты берёшь каждую тетрадь, проверяешь её (вызываешь первую функцию) и записываешь сколько ошибок
- Потом считаешь **среднее количество ошибок** на одну тетрадь

**Почему именно так:**

1. **Проверка пачками (batch), а не по одному примеру:**
   - Это быстрее! Как проверять сразу несколько тетрадей, а не по одной
   - Компьютер хорошо умеет делать много одинаковых действий параллельно

2. **Можно проверить не всё (`num_batches`):**
   - Иногда данных очень много, проверка всего занимает часы
   - Можно проверить только часть, чтобы быстро понять, как идёт обучение

3. **`float("nan")` если данных нет:**
   - Это как сказать "невозможно посчитать", если тетрадей вообще нет
   - Защита от ошибки деления на ноль

4. **`loss.item()` вместо просто `loss`:**
   - PyTorch хранит числа в специальной "умной" упаковке
   - `.item()` достаёт простое число, которое можно складывать в корзинку

5. **Среднее значение в конце (`total_loss / num_batches`):**
   - Честное сравнение: неважно, проверили мы 10 пачек или 100
   - Всегда получаем среднюю ошибку на одну пачку
   
### Простая аналогия: 

Представь, что ты учишь попугая говорить:

- **Первая функция** — это когда ты показываешь попугаю 32 слова и проверяешь, сколько из них он повторил неправильно
- **Вторая функция** — это когда ты проверяешь его целый день (или час), записываешь все ошибки, а потом считаешь "в среднем 5 ошибок за раз"

Если с каждым днём среднее количество ошибок уменьшается — попугай учится! 🦜

 ---

- Если у вас есть компьютер с графическим процессором, поддерживающим CUDA, LLM будет обучаться на графическом процессоре без каких-либо изменений в коде
- С помощью параметра `device` мы гарантируем, что данные будут загружены на то же устройство, что и модель LLM

In [30]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # Для стабильных результатов используйте PyTorch 2.9 или более новую версию
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

print(f"Используется устройство: {device}.")

model.to(device)  # для классов nn.Module не требуется присваивание model = model.to(device)

torch.manual_seed(123)  # Для воспроизводимости результатов из-за перемешивания в загрузчике данных

with torch.no_grad():  # Отключаем отслеживание градиентов для эффективности, так как модель пока не обучается
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

print("Потери на обучающей выборке:", train_loss)
print("Потери на проверочной выборке:", val_loss)

Используется устройство: cpu.
Потери на обучающей выборке: 10.98758347829183
Потери на проверочной выборке: 10.981106758117676


<img src="https://camo.githubusercontent.com/de4c90a17bebd1ed30b5ae1724724d2fb8fec8233258eb2c6c931e17fb3a3b5a/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f31302e77656270" width=800px>